# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a git pull and merge your changes as needed</a>. Check out the GitHub guide for instructions. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

### Importing the libraries

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openrouter import OpenRouter
from google import genai
from openai import OpenAI
from IPython.display import display, Markdown

In [ ]:
## loading the api keys

load_dotenv(override=True)

open_router_api_key = os.getenv('OPENROUTER_API_KEY')
gemini_api_key = os.getenv('GOOGLE_API_KEY')
# deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if (open_router_api_key and open_router_api_key.startswith('sk-or')):
    print('Open Router API key found and good so far')
else:
    print('Api Key not found')

if gemini_api_key:
    print(f"Google API Key exists and begins {gemini_api_key[:2]}")
else:
    print("Google API Key not set")

# if deepseek_api_key:
#     print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
# else:
#     print("DeepSeek API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# deepseek_url = "https://api.deepseek.com"

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
open_router_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
open_router = OpenAI(api_key=open_router_api_key, base_url=open_router_url)
ollama = OpenAI(api_key='ollama', base_url=ollama_url)


### OpenRouter Free Models

In [ ]:
gemini_model_gemma_4_31b = 'google/gemma-4-31b-it:free'

gpt_oss_model = 'openai/gpt-oss-120b:free'

# Below models are now converted to paid ones

# inclusionai_model_ring_2_1t = 'inclusionai/ring-2.6-1t:free'

# baidu_model = "baidu/cobuddy:free"

# deepseek_v4_flash_model = "deepseek/deepseek-v4-flash:free"

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [ ]:
def call_openai(client_llm, model, messages, **kwargs):
    response = client_llm.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs
    )
    return response.choices[0].message.content


In [ ]:
result = call_openai(gemini, 'gemini-2.5-flash-lite', tell_a_joke)
display(Markdown(result))

In [ ]:
result = call_openai(open_router, gpt_oss_model, tell_a_joke)
display(Markdown(result))

## Training vs Inference time scaling

In [ ]:
easy_puzzle = [
    {"role": "user", "content": 
    "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."}
]

In [ ]:
result = call_openai(gemini, 'gemini-3.1-flash-lite', easy_puzzle)
display(Markdown(result))

In [ ]:
result = call_openai(open_router, gpt_oss_model, easy_puzzle, reasoning_effort='low')
display(Markdown(result))

## Testing out with hard problems

In [ ]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
Do not respond in code blocks. Think Step by Step
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
result = call_openai(open_router, gpt_oss_model, hard_puzzle, reasoning_effort='medium')
display(Markdown(result))

In [ ]:
result = call_openai(gemini, 'gemini-3.1-flash-lite', hard_puzzle)
display(Markdown(result))

## A spicy challenge to test the competitive spirit

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]

In [ ]:
result = call_openai(open_router, gpt_oss_model, dilemma)
display(Markdown(result))

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
result = call_openai(ollama, 'llama3.2:1b', easy_puzzle)
display(Markdown(result))

### Gemini Endpoint Library

In [ ]:
client = genai.Client()

def call_gemini(model):
    response = client.models.generate_content(
        model=model,
        contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
    )
    print(f'Below is the response from the model {model}')
    print(response.text)

In [ ]:
call_gemini('gemini-2.5-flash-lite')

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.


In [ ]:
response = call_openai(open_router, baidu_model, tell_a_joke)
display(Markdown(result))

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [ ]:
## Below is just the code, but it won't work, since I don't have money for OpenAI API :/

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5-nano')
response = llm.invoke(tell_a_joke)

display(Markdown(response))

In [ ]:
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model="gemini-2.5-flash-lite")
response = llm.invoke(tell_a_joke)

display(Markdown(response))

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [ ]:
from litellm import completion

In [ ]:
response = completion(model='gemini/gemini-3.1-flash-lite', messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
# Usage(completion_tokens=92, prompt_tokens=18, total_tokens=110, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=None, rejected_prediction_tokens=None, text_tokens=92, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=None, text_tokens=18, image_tokens=None, video_tokens=None), cache_read_input_tokens=None)

print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Total tokens: {response.usage.total_tokens}')
print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


## Now - let's use LiteLLM to illustrate a Pro-feature: prompt caching

In [ ]:
with open('hamlet.txt', 'r') as fs:
    hamlet = fs.read()

loc = hamlet.find('Speak, man')
print(hamlet[loc:loc+100])

In [ ]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [ ]:
response = completion(api_key=open_router_api_key, base_url=open_router_url, model=f'openai/{gpt_oss_model}', messages=question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:

print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Total tokens: {response.usage.total_tokens}')
print(f'Total cost: {response._hidden_params["response_cost"]} cents')

# print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


In [ ]:
question[0]['content'] += f"\n\nFor Context here is the entire text of Hamlet\n\n"+hamlet

In [ ]:
response = completion(api_key=open_router_api_key, base_url=open_router_url, model=f'openai/{gpt_oss_model}', messages=question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:

print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}')

# print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


### Calling the LLM using same prompt - prompt caching

In [ ]:
response = completion(api_key=open_router_api_key, base_url=open_router_url, model=f'openai/{gpt_oss_model}', messages=question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:

print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}')

# print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


In [ ]:
new_question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

### Testing Gemini 2.5 flash lite using hamlet question

In [ ]:
response = completion(model='gemini/gemini-2.5-flash-lite', messages=new_question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Total tokens: {response.usage.total_tokens}')

In [ ]:
new_question[0]['content'] += f"\n\nFor Context here is the entire text of Hamlet\n\n"+hamlet

In [ ]:
response = completion(model='gemini/gemini-2.5-flash-lite', messages=new_question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}')
print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


In [ ]:
response = completion(model='gemini/gemini-2.5-flash-lite', messages=new_question)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f'Input tokens: {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
print(f'Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}')
print(f'Total cost: {response._hidden_params["response_cost"]*100:.4f} cents')


## Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

## Gemini supports both 'implicit' and 'explicit' prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [ ]:
# Conversation between gpt oss and deepseek v4 flash
# gpt_oss_model, deepseek_v4_flash_model

gpt_sytem = "You are a chatbot who is very arugmentative; \
you disagree with anything in the conversation and challenge everything in a snarky way."

deepseek_sytem = """
You are a very polite, courteous chatbot. You try to agree with 
everything the other person says, or find common ground. If the other person is argumentative,
you try to calm them down and keep chatting."
"""

gpt_messages = ["Hi there"]
deepseek_messages = ["Hi"]


In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_sytem}]
    for gpt, deepseek in zip(gpt_messages, deepseek_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": deepseek})
    response = call_openai(open_router, gpt_oss_model, messages=messages)
    return response


In [ ]:
def call_deepseek():
    messages = [{"role": "system", "content": deepseek_sytem}]
    for gpt, deepseek in zip(gpt_messages, deepseek_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": deepseek})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = call_openai(open_router, deepseek_v4_flash_model, messages=messages)
    return response


In [ ]:
gpt_messages = ["Hi there"]
deepseek_messages = ["Hi"]

display(Markdown(f'### ChatGPT \n{gpt_messages[0]}\n'))
display(Markdown(f'### DeepSeek \n{deepseek_messages[0]}\n'))

for i in range(5):
    gpt_text = call_gpt()
    display(Markdown(f'\n ### ChatGPT:{gpt_text}\n\n'))
    gpt_messages.append(gpt_text)

    deepseek_text = call_deepseek()
    display(Markdown(f'\n ### DeepSeek:{deepseek_text}\n\n'))
    deepseek_messages.append(deepseek_text)


# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business relevance</h2>
            <span style="color:#181;">This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.</span>
        </td>
    </tr>
</table>

In [ ]:
## ChatBots, 3 way conversation.

## Defining the personas

system_prompt_alex = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie. Keep replies under 2 sentences.
"""

system_prompt_blake = """
You are Blake, a chatbot who is very polite; you agree with anything in the conversation and you do not challenge anything.
You are in a conversation with Alex and Charlie. Keep replies under 2 sentences.
"""

system_prompt_charlie = """
You are Charlie, a chatbot who is neutral, you can agree or disagree with anything in the conversation, but you remain neutral in the end for
everything in the conversation.
You are in a conversation with Alex and Blake. Keep replies under 2 sentences.
"""

conversation = [('Alex', 'Hello'), ('Blake', 'Hi There'), ('Charlie', 'Hii')]


In [ ]:
def call_alex():
    messages = [{"role": "system", "content": system_prompt_alex}]
    convo = ''
    for person, mssg in conversation:
        convo += f'{person}: {mssg} \n'
    
    user_prompt_alex = f"""
    You are Alex, in conversation with Blake and Charlie.
    The conversation so far is as follows:
    {convo}
    Now with this, respond with what you would like to say next, as Alex.
    """
    messages.append({"role": "user", "content": user_prompt_alex})
    alex_response = call_openai(open_router, gpt_oss_model, messages=messages)
    return alex_response


In [ ]:
def call_blake():
    messages = [{"role": "system", "content": system_prompt_blake}]
    convo = ''
    for person, mssg in conversation:
        convo += f'{person}: {mssg} \n'
    
    user_prompt_blake = f"""
    You are Blake, in conversation with Alex and Charlie.
    The conversation so far is as follows:
    {convo}
    Now with this, respond with what you would like to say next, as Blake.
    """
    messages.append({"role": "user", "content": user_prompt_blake})
    blake_response = call_openai(open_router, gpt_oss_model, messages=messages)
    return blake_response


In [ ]:
def call_charlie():
    messages = [{"role": "system", "content": system_prompt_charlie}]
    convo = ''
    for person, mssg in conversation:
        convo += f'{person}: {mssg} \n'
    
    user_prompt_charlie = f"""
    You are Charlie, in conversation with Alex and Blake.
    The conversation so far is as follows:
    {convo}
    Now with this, respond with what you would like to say next, as Charlie.
    """
    messages.append({"role": "user", "content": user_prompt_charlie})
    charlie_response = call_openai(open_router, gpt_oss_model, messages=messages)
    return charlie_response


In [ ]:
for i in range(3):
    display(Markdown(f'\n ### {conversation[i][0]}: {conversation[i][1]}\n'))

# calling all personas 3 times 

def start_chat_bots(n):
    for i in range(n):
        alex_text = call_alex()
        display(Markdown(f'\n ### Alex: {alex_text}\n'))
        conversation.append(('Alex', alex_text))

        blake_text = call_blake()
        display(Markdown(f'\n ### Blake: {blake_text}\n'))
        conversation.append(('Blake', blake_text))

        charlie_text = call_charlie()
        display(Markdown(f'\n ### Charlie: {charlie_text}\n'))
        conversation.append(('Charlie', charlie_text))

start_chat_bots(4)
